# Задача 5

## Предистория
Водеща компания за социални медии е разработила сложна система за класификация на съдържанието, която помага за организиране и категоризиране на генерираното от потребителите съдържание в тяхната платформа. Най-новата им технология за изкуствен интелект произвежда висококачествени вграждания за текстово съдържание (което ще симулираме с помощта на вграждания Sentence-BERT в тази задача).

## Предизвикателството
Въпреки че настоящата система работи отлично в сървърна инфраструктура, компанията има за цел да пренесе част от тези възможности за класификация директно на мобилните устройства на потребителите. Това изисква значително намаляване на размерите на вграждането, като същевременно се запазва възможно най-голяма част от първоначалната способност за клъстериране.

## Вашата задача
Вашата задача е да разработите функция за преобразуване, която може да преобразува оригиналните 384-измерни вграждания в 32-измерни вграждания, като същевременно запазва основната информация, необходима за точното клъстеризиране на съдържанието.

## Правила
- Не променяйте предоставените клетки, различни от отбелязаните за вашата реализация.
- Обучението на модели е разрешено, но директното използване на K-Means или KNN във функцията за преобразуване е *забранено*.
- Можете да оценявате с по-малко или повече от 10 изпълнения, но окончателните заявки ще бъдат оценени с 10 изпълнения.
- Подсказка: Опитайте се да направите решението си възможно най-стабилно на случайни инициализации.

## Предаване
Предайте архив, съдържащ тази тетрадка и придружаващи файлове, ако смятате за нужно - имената на тези файлове трябва да съдържат вашите имена.



# Setup

In [1]:
!pip install gdown
!gdown 1pDf-ZnqJdOZYHWj0Rp4Y6nOoHLbzuTSt
!gdown 1aVT0ANgCTrMLXwU5MXy2y_BUyLwPba7t

Downloading...
From: https://drive.google.com/uc?id=1pDf-ZnqJdOZYHWj0Rp4Y6nOoHLbzuTSt
To: /content/public_texts.pkl
100% 213k/213k [00:00<00:00, 7.84MB/s]
Downloading...
From: https://drive.google.com/uc?id=1aVT0ANgCTrMLXwU5MXy2y_BUyLwPba7t
To: /content/private_texts.pkl
100% 212k/212k [00:00<00:00, 4.92MB/s]


In [2]:
import os
import torch
import pickle
import random
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from sklearn.datasets import fetch_20newsgroups
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

In [3]:
def set_seed(seed_value = 42):
  random.seed(seed_value)
  np.random.seed(seed_value)
  torch.manual_seed(seed_value)
  if torch.cuda.is_available():
      torch.cuda.manual_seed(seed_value)
      torch.cuda.manual_seed_all(seed_value)
      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

set_seed()

In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, model_name='all-MiniLM-L6-v2'):
        self.texts = texts
        self.labels = labels
        self.model = SentenceTransformer(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        with torch.no_grad():
            emb = self.model.encode(text, convert_to_tensor=True)
        label = self.labels[idx]
        return emb, label

In [5]:
def load_and_process_split(split_name, model_name='all-MiniLM-L6-v2'):
    with open(f"{split_name}_texts.pkl", "rb") as f:
        texts, labels = pickle.load(f)

    dataset = TextDataset(texts, labels, model_name=model_name)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    X = []
    y = []
    for emb, label in tqdm(loader):
        X.append(emb.cpu().numpy())
        y.append(label.numpy())

    return np.vstack(X), np.concatenate(y)

# Main code

In [9]:
X384, y = load_and_process_split("public")
X384_test, y = load_and_process_split("private")
k = 4

100%|██████████| 125/125 [00:24<00:00,  5.02it/s]


In [11]:
set_seed()
km384 = KMeans(n_clusters=k, random_state=0).fit(X384)
public_clusters = km384.labels_
private_clusters = km384.predict(X384_test)

In [12]:
from sklearn.decomposition import PCA
import numpy as np
pca_model = PCA(n_components=32)
pca_model.fit(X384)
def transform(X_high):
    assert X_high.ndim == 2 and X_high.shape[1] >= 32, "Input must be 2D with >=32 dims."
    ### TODO: Добавете своя код тук
    out = pca_model.transform(X_high)
    ###
    assert out.ndim == 2 and out.shape[1] == 32, f"Embeddings must be shape (N,32), got {out.shape}."
    return out

In [14]:
def evaluate_clustering(ref_labels, n_clusters, n_runs=10):
    nmi_scores = []
    for seed in range(n_runs):
        set_seed(seed)
        X32 = transform(X384_test)
        km = KMeans(n_clusters=n_clusters, random_state=0).fit(X32)
        pred = km.labels_
        nmi = normalized_mutual_info_score(ref_labels, pred)
        nmi_scores.append(nmi)

    mean_nmi = np.mean(nmi_scores)
    std_nmi = np.std(nmi_scores)
    return mean_nmi, std_nmi

mean_nmi, std_nmi = evaluate_clustering(private_clusters, k)
print(f"\nPublic NMI (first-32 dims): {mean_nmi:.4f} ± {std_nmi:.4f}")


Public NMI (first-32 dims): 0.8727 ± 0.0000


In [ ]:
def evaluate_clustering(ref_labels, n_clusters, n_runs=10):
    nmi_scores = []
    for seed in range(n_runs):
        set_seed(seed)
        X32 = transform(X384)
        km = KMeans(n_clusters=n_clusters, random_state=0).fit(X32)
        pred = km.labels_
        nmi = normalized_mutual_info_score(ref_labels, pred)
        nmi_scores.append(nmi)

    mean_nmi = np.mean(nmi_scores)
    std_nmi = np.std(nmi_scores)
    return mean_nmi, std_nmi

mean_nmi, std_nmi = evaluate_clustering(public_clusters, k)
print(f"\nPublic NMI (first-32 dims): {mean_nmi:.4f} ± {std_nmi:.4f}")


Public NMI (first-32 dims): 0.2487 ± 0.0000


## Разбиране на метриката за оценка: NMI

Нормализираната взаимна информация (NMI) е мярка, която ни показва доколко две различни групирания (клъстерирания) съвпадат едно с друго. Тя е особено подходяща за това предизвикателство по няколко причини:

- **Независимост от мащаба**: NMI е нормализирана стойност между 0 (никаква взаимна информация) и 1 (перфектна корелация), което я прави лесна за интерпретиране, независимо от броя на клъстерите или точките с данни.

- **Инвариантност спрямо пермутации**: NMI не изисква етикетите на клъстерите да съвпадат точно – тя се интересува само от цялостната структура на групиране. Това е важно, тъй като k-means може да присвои различни цифрови етикети на едни и същи логически клъстери при различни изпълнения.

В нашия случай използваме NMI, за да сравним:
- Клъстерирането, получено от оригиналните 384-измерни ембединги (референтно).
- Клъстерирането, получено от вашите трансформирани 32-измерни ембединги.

По-високият NMI резултат означава, че вашата трансформация по-добре запазва оригиналната структура на клъстериране, което е точно това, което целим за сценария с внедряване на мобилни устройства.
